In [2]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch.utils.data import DataLoader, Dataset
from typing import Optional
import tqdm as tqdm

## `things to do:`
`def __init__():` adjust kinematic and occupancy parameters, adapt event list names   
`def load_event():` and `def __getitem__():` need to load whole file into constructor and extract data like that. also need to adjust what parameters we calculate from given parameters

`def load_event():` is done
`def __getitem__():` needs to be fully completed


dont wanna apply any cuts - most cuts are implicit in MC  
also group tracks by mu3e tree eventID rather than fake ID  
check pdg + mc_track on mc_hits - do this without making cut on hits with NaN eventID  
probs dont need to bother ab volume_ids - check w nikita  
check what kinematic particle cuts actually do  
think ab any other important variables that might be useful  
could apply lambda/p_t cuts  

In [2]:
class Mu3eDataset(Dataset):
    # sets up PyTorch dataset for TrackML data, which events to load, how to filter hits/tracks, handles dummy test data
    def __init__(
        self,
        dirpath: str,
        inputs: dict,            # dictionary specifying which hit features to include in input tensors
        targets: dict,           # dictionary specifying which features should be inlcuded as model targets
        num_events: int = -1,
        hit_volume_ids: Optional[list] = None,
        feature_volume_ids: Optional[dict] = None,
        #particle_min_pt: float = 1.0,                   # minimum transverse momentum for particles to keep
        #particle_max_abs_eta: float = 2.5,              # maximum psuedorapidity for particles to keep
        particle_min_num_hits: Optional[int] = None,    # minimum number of hits a particle must have to keep
        event_max_num_particles=1000,                   # maximum number of hits 
        strict_max_objects: bool = False,
        hit_eval_path: Optional[str] = None,
        dummy_data: bool = False,
    ):
        super().__init__()

        # Store dummy_data flag
        self.dummy_data = dummy_data

        # Set the global random sampling seed
        self.sampling_seed = 42
        np.random.seed(self.sampling_seed)  # noqa: NPY002

        # If using dummy data, skip file-based initialization
        if self.dummy_data:
            rank_zero_info("Generating dummy data...")
            self.dirpath = Path(dirpath) if dirpath else Path()
            self.hit_eval_path = None
            self.inputs = inputs
            self.targets = targets
            self.num_events = max(num_events, 1) if num_events > 0 else 10
            self.event_names = [f"dummy_event_{i:06d}" for i in range(self.num_events)]
            self.sample_ids = list(range(self.num_events))
            self.hit_volume_ids = hit_volume_ids
            #self.particle_min_pt = particle_min_pt
            #self.particle_max_abs_eta = particle_max_abs_eta
            self.particle_min_num_hits = particle_min_num_hits
            self.event_max_num_particles = event_max_num_particles
            return
        
        # list of event names for mu3e parquet files
        track_file_path = Path(dirpath) / 'all_hits.parquet'
        event_id_series = pq.read_table(track_file_path, columns=['eventID']).to_pandas()['eventID']
        # get unique and sorted eventIDs
        unique_event_ids = event_id_series.unique()
        unique_event_ids.sort()
        # create standarised event names and sample ids
        event_names = [f'event{ID:09d}' for ID in unique_event_ids]
        sample_ids = unique_event_ids.tolist()

        # Calculate the number of events that will actually be used
        num_events_available = len(event_names)

        # sanity checks 
        if num_events > num_events_available:
            msg = f"Requested {num_events} events, but only {num_events_available} are available in the directory {dirpath}."
            raise ValueError(msg)

        if num_events < 0:
            num_events = num_events_available

        if num_events == 0:
            raise ValueError("num_events must be greater than 0")

        # Metadata
        self.dirpath = Path(dirpath)
        self.hit_eval_path = hit_eval_path
        self.inputs = inputs
        self.targets = targets
        self.num_events = num_events
        self.event_names = event_names[:num_events]
        self.sample_ids = sample_ids[:num_events]

        # Setup hit eval file if specified
        if self.hit_eval_path:
            rank_zero_info(f"Using hit eval dataset {self.hit_eval_path}")

        # store filtering parameters
        # Hit level cuts
        self.hit_volume_ids = hit_volume_ids
        # Optional per-feature hit volume selections
        self.feature_volume_ids = feature_volume_ids

        # Particle level cuts
        #self.particle_min_pt = particle_min_pt
        #self.particle_max_abs_eta = particle_max_abs_eta
        self.particle_min_num_hits = particle_min_num_hits

        # Event level cuts
        self.event_max_num_particles = event_max_num_particles
        self.strict_max_objects = strict_max_objects

    def __len__(self):
        return int(self.num_events)

In [3]:
class Mu3eDataset(Dataset): 
    # what makes it work like a PyTorch dataset - give index, returns data for the sample (event)
    def __getitem__(self, idx):
        if self.dummy_data:
            return self._generate_dummy_data(idx)

        # prepare containers
        inputs = {}
        targets = {}

        # Load the event
        hits, tracks = self.load_event(idx)
        num_tracks = len(tracks)

        # Build the input hits
        for feature, fields in self.inputs.items():
            feature_hits = hits

            # Valid mask is all True for the feature-specific subset
            inputs[f"{feature}_valid"] = torch.full((len(feature_hits),), True).unsqueeze(0)
            targets[f"{feature}_valid"] = inputs[f"{feature}_valid"]
            
            for field in fields:
                inputs[f"{feature}_{field}"] = torch.from_numpy(feature_hits[field].values).unsqueeze(0).half()

        # Create the targets for whether a particle slot is used or not
        if num_particles > self.event_max_num_particles:
            if self.strict_max_objects:
                message = f"Event {idx} has {num_particles}, but limit is {self.event_max_num_particles}"
                raise ValueError(message)
            tracks = tracks.iloc[: self.event_max_num_particles]
            num_particles = self.event_max_num_particles

        # Create particle_valid mask by concatenating True and False arrays
        num_padding = self.event_max_num_particles - num_particles
        targets["particle_valid"] = torch.cat([torch.full((num_particles,), True), torch.full((num_padding,), False)]).unsqueeze(0)

        # Create the mask targets
        selected_track_ids = torch.from_numpy(tracks["trackID"].values)
        track_ids = torch.cat([selected_track_ids, torch.full((num_padding,), -999)])
        hit_track_ids = torch.from_numpy(hits["trackID"].values)
        targets["track_hit_valid"] = (track_ids.unsqueeze(-1) == hit_track_ids.unsqueeze(-2)).unsqueeze(0)

        # Create the hit filter targets (note this ignores the event_max_num_particles filtering)
        for target_feature, fields in self.targets.items():
            if "on_valid_particle" in fields:
                targets[f"{target_feature}_on_valid_particle"] = torch.from_numpy(hits["on_valid_particle"].to_numpy()).unsqueeze(0)

        # Add sample ID
        targets["sample_id"] = torch.tensor([self.sample_ids[idx]], dtype=torch.int32)

        # Build the regression targets
        if "particle" in self.targets:
            for field in self.targets["particle"]:
                # Null target/particle slots are filled with nans
                x = torch.full((self.event_max_num_particles,), torch.nan)
                x[:num_particles] = torch.from_numpy(particles[field].to_numpy()[: self.event_max_num_particles])
                targets[f"particle_{field}"] = x.unsqueeze(0)

        return inputs, targets

In [3]:
class Mu3eDataset(Dataset):
    def load_event(self, idx):
        sample_id = self.sample_ids[idx]
        event_name = self.event_names[idx]

        # load data for the specific event
        all_hits = pd.read_parquet(self.dirpath / Path("all_hits.parquet"))
        all_tracks = pd.read_parquet(self.dirpath / Path("all_tracks.parquet"))
        
        hits = all_hits[all_hits['eventID'] == sample_id]
        tracks = all_tracks[all_tracks['eventID'] == sample_id]
        
        # make the detector volume selection
        if self.hit_volume_ids:
            hits = hits[hits["volume_id"].isin(self.hit_volume_ids)]

        # add extra hit fields (geometric features)
        hits["r"] = np.sqrt(hits["x"] ** 2 + hits["y"] ** 2)
        hits["s"] = np.sqrt(hits["x"] ** 2 + hits["y"] ** 2 + hits["z"] ** 2)
        hits["lambda"] = np.arccos(hits["z"] / hits["s"])                       # use our variable lambda
        hits["phi"] = np.arctan2(hits["y"], hits["x"])
        hits["eta"] = -np.log(np.tan(hits["theta"] / 2))
        hits["u"] = hits["x"] / (hits["x"] ** 2 + hits["y"] ** 2)
        hits["v"] = hits["y"] / (hits["x"] ** 2 + hits["y"] ** 2)

        # add extra track fields (kinematic features)
        tracks["p"] = np.sqrt(tracks["px"] ** 2 + tracks["py"] ** 2 + tracks["pz"] ** 2)
        tracks["pt"] = np.sqrt(tracks["px"] ** 2 + tracks["py"] ** 2)
        tracks["eta"] = np.arctanh(tracks["pz"] / tracks["p"])
        tracks["lambda"] = np.arccos(tracks["pz"] / tracks["p"])
        tracks["phi"] = np.arctan2(tracks["py"], tracks["px"])
        tracks["costheta"] = np.cos(tracks["theta"])
        tracks["sintheta"] = np.sin(tracks["theta"])
        tracks["cosphi"] = np.cos(tracks["phi"])
        tracks["sinphi"] = np.sin(tracks["phi"])

        # apply particle level cuts based on track fields (kinematic cuts)  # not sure if necessary ### 
        #tracks = tracks[tracks["pt"] > self.particle_min_pt]
        #tracks = tracks[tracks["eta"].abs() < self.particle_max_abs_eta]

        ########################################### idk what this does - talk to nikita #####################################################
        # If a hit eval file was specified, read in the predictions from it to use the hit filtering
        #if self.hit_eval_path:
        #    with h5py.File(self.hit_eval_path, "r") as hit_eval_file:
        #        assert str(self.sample_ids[idx]) in hit_eval_file, f"Key {self.sample_ids[idx]} not found in file {self.hit_eval_path}"#

                # The dataset has shape (1, num_hits)
        #        hit_filter_pred = hit_eval_file[f"{self.sample_ids[idx]}/preds/final/hit_filter/hit_on_valid_particle"][0]
        #        hits = hits[hit_filter_pred]
        ########################################### idk what this does - talk to nikita #####################################################

        # apply particle cut based on hit content (minimum hits / naked tracks)
        counts = hits["trackID"].value_counts()
        keep_track_ids = counts[counts >= self.particle_min_num_hits].index.to_numpy()
        tracks = tracks[tracks["trackID"].isin(keep_track_ids)].reset_index(drop=True)

        # mark which hits are on a valid / reconstructable particle, for the hit filter
        hits["on_valid_particle"] = hits["trackID"].isin(tracks["trackID"])

        # sanity checks
        assert len(tracks) != 0, "No particles remaining - loosen selection!"
        assert len(hits) != 0, "No hits remaining - loosen selection!"
        assert track["trackID"].nunique() == len(tracks), "Non-unique particle ids"

        return hits, tracks

In [ ]:
class Mu3eDataModule(LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        val_dir: str,
        num_workers: int,
        num_train: int,
        num_val: int,
        num_test: int,
        test_dir: str | None = None,
        pin_memory: bool = True,
        hit_eval_train: str | None = None,
        hit_eval_val: str | None = None,
        hit_eval_test: str | None = None,
        **kwargs,
    ):
        super().__init__()

        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.num_workers = num_workers
        self.num_train = num_train
        self.num_val = num_val
        self.num_test = num_test
        self.pin_memory = pin_memory
        self.hit_eval_train = hit_eval_train
        self.hit_eval_val = hit_eval_val
        self.hit_eval_test = hit_eval_test
        self.kwargs = kwargs

    def setup(self, stage: str):
        if stage in {"fit", "test"}:
            self.train_dataset = TrackMLDataset(
                dirpath=self.train_dir,
                num_events=self.num_train,
                hit_eval_path=self.hit_eval_train,
                **self.kwargs,
            )

        if stage == "fit":
            self.val_dataset = TrackMLDataset(
                dirpath=self.val_dir,
                num_events=self.num_val,
                hit_eval_path=self.hit_eval_val,
                **self.kwargs,
            )

        # Only print train/val dataset details when actually training
        if stage == "fit":
            rank_zero_info(f"Created training dataset with {len(self.train_dataset):,} events")
            rank_zero_info(f"Created validation dataset with {len(self.val_dataset):,} events")

        if stage == "test":
            assert self.test_dir is not None, "No test file specified, see --data.test_dir"

            self.test_dataset = TrackMLDataset(
                dirpath=self.test_dir,
                num_events=self.num_test,
                hit_eval_path=self.hit_eval_test,
                **self.kwargs,
            )
            rank_zero_info(f"Created test dataset with {len(self.test_dataset):,} events")

    def get_dataloader(self, stage: str, dataset: TrackMLDataset, shuffle: bool):
        return DataLoader(
            dataset=dataset,
            batch_size=None,
            collate_fn=None,
            sampler=None,
            num_workers=self.num_workers,
            shuffle=shuffle,
            pin_memory=self.pin_memory,
        )

    def train_dataloader(self):
        return self.get_dataloader(dataset=self.train_dataset, stage="fit", shuffle=True)

    def val_dataloader(self):
        return self.get_dataloader(dataset=self.val_dataset, stage="test", shuffle=False)

    def test_dataloader(self):
        return self.get_dataloader(dataset=self.test_dataset, stage="test", shuffle=False)